In [1]:
import sys
sys.path.append("../preprocess")
from qbiolip import load_db
import glob
import torch
import re
from plotly import express as xp
from plotly import graph_objects as go
import pandas

df = load_db("../../databases/QBioLiP/qbiolip.csv", "../../databases/ec_uniprot_ids.txt")
df = df.loc[df.target]
df["diffdock_rmsd"] = None
df["diffdock_trimed_rmsd"] = None

# organize the outputs so metrics can be calculated:
output_folders = glob.glob("/home/daniil/Desktop/Ph.D/DiffDock/results/user_inference/*")
preds = []
targets = []
names = []
confidences = []
for folder in output_folders:
    # get the target xyz coordinates:
    name = folder.split("/")[-1]
    target_path = f"../../databases/QBioLiP/nr_lig/{name.replace('_trimed', '')}.sdf"
    with open(target_path) as target_f:
        lines = [l.split() for l in target_f]
    lines = [l for l in lines if len(l) == 16] #only xyz coordinates lines
    target_xyz = torch.tensor([
        [
            float(l[0]), #x
            float(l[1]), #y
            float(l[2]), #z
        ]
    for l in lines])
    targets.append(target_xyz)
    # get prediction data:   
    names.append(name)
    folder_outputs = glob.glob(f"{folder}/*.sdf")
    folder_confidences = []
    folder_predicted = []
    for sdf_path in folder_outputs:
        # check if confidence is available:
        confidence_match = re.search(r"([-]|(?<=e))\d.\d{1,2}", sdf_path.split("/")[-1])
        if confidence_match is None:
            continue
        confidence = float(confidence_match.group())
        folder_confidences.append(confidence)
        # get the predicted xyz:
        with open(sdf_path) as sdf_f:
            lines = [l.split() for l in sdf_f]
        lines = [l for l in lines if len(l) == 16] #only xyz coordinates lines
        xyz_predicted = [
            [
                float(l[0]), #x
                float(l[1]), #y
                float(l[2]), #z
            ]
        for l in lines]
        if len(xyz_predicted) != target_xyz.shape[0]:
            folder_confidences = folder_confidences[:-1]
            continue
        folder_predicted.append(xyz_predicted)
    # rank the prediction based on the confidence
    folder_confidences = torch.tensor(folder_confidences)
    folder_predicted = torch.tensor(folder_predicted)
    ranks = folder_confidences.argsort(descending=True) # the higher the better
    folder_predicted = folder_predicted[ranks]
    preds.append(folder_predicted)
    confidences.append(folder_confidences)

# calculate metrics:
rank = 0 #look at the highest confidence score
print(len(names))
for i in range(len(names)):
    if preds[i].shape[0] == 0:
        continue
    pred = preds[i][rank]
    target = targets[i]
    rmsd = (pred-target).pow(2).sum(dim=1).mean().sqrt()
    name = names[i]
    col = ("diffdock_rmsd", "diffdock_trimed_rmsd")["_trimed" in name]
    df.loc[df.ligand_detail == name.replace("_trimed", ""), [col]] = rmsd
print("Number of available rmsd for full receptor:", df.loc[~df.diffdock_rmsd.isna()].shape[0])
print("Number of available trimed rmsd:",df.loc[~df.diffdock_trimed_rmsd.isna()].shape[0])

# plot metrics:
diffdock_df = df.loc[(~df.diffdock_rmsd.isna()) & (~df.diffdock_trimed_rmsd.isna())]

# plot untrimed:
diffdock_df.sort_values(by="diffdock_rmsd", inplace=True)
diffdock_df.reset_index(drop=True,inplace=True)
colors = pandas.Series(["red"]*diffdock_df.shape[0])
colors[diffdock_df.loc[diffdock_df.group=="B"].index] = "blue"
colors[diffdock_df.loc[diffdock_df.group=="C"].index] = "green"

fig = xp.bar(
    diffdock_df, x="ligand_detail", y="diffdock_rmsd",
    title="DiffDock RMSD on 90 enzyme-ligand complexes with implicit hydrogenation, true blind docking, top 1",
    color=colors,
    color_discrete_map="identity",
    category_orders={"ligand_detail": diffdock_df.ligand_detail.tolist()},
    log_y=True,
)
fig.update_layout(
    xaxis_title="Name of the complex",
    yaxis_title="RMSD, Logarithmic scale",
)
fig.show()

# plot trimed
diffdock_df.sort_values(by="diffdock_trimed_rmsd", inplace=True)
diffdock_df.reset_index(drop=True,inplace=True)
colors = pandas.Series(["red"]*diffdock_df.shape[0])
colors[diffdock_df.loc[diffdock_df.group=="B"].index] = "blue"
colors[diffdock_df.loc[diffdock_df.group=="C"].index] = "green"

fig = xp.bar(
    diffdock_df, x="ligand_detail", y="diffdock_trimed_rmsd",
    title="DiffDock RMSD on 90 enzyme-ligand complexes with implicit hydrogenation, pseudo blind docking, top 1",
    color=colors,
    color_discrete_map="identity",
    category_orders={"ligand_detail": diffdock_df.ligand_detail.tolist()},
    log_y=True,
)
fig.update_layout(
    xaxis_title="Name of the complex",
    yaxis_title="RMSD, Logarithmic scale",
)
fig.show()
# plot rmsd per group # trimed
x = diffdock_df.group.unique()
y_trimed = [diffdock_df.loc[diffdock_df.group==group].diffdock_trimed_rmsd.mean() for group in x]
# plot rmsd per group # not trimed
y_untrimed = [diffdock_df.loc[diffdock_df.group==group].diffdock_rmsd.mean() for group in x]
fig = go.Figure([
    go.Bar(x=x, y=y_trimed, name="trimed"),
    go.Bar(x=x, y=y_untrimed, name="untrimed"),
])
fig.update_layout(
    title="Mean RMSD for each categories"
)
fig.show()

Len of the database: 25415
Columns: id, pdb_id, assembly_id, stoichiometry, resolution, uniprot_id, assembly_detail, ligand_id, site, ligand_detail, pubmed_id, interaction, relevant, binding_site, binding_site_pdb, bindingMOAD, pdbbind_cn, binding_db, group, lig_canon_smiles, lig_atoms, rec_len_resi, target, trimed_rec_path
180
Number of available rmsd for full receptor: 61
Number of available trimed rmsd: 81


/tmp/ipykernel_88723/1869841713.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diffdock_df.sort_values(by="diffdock_rmsd", inplace=True)


/tmp/ipykernel_88723/1869841713.py:112: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

